# 1. Carga de datos

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

esqueleto = pd.read_parquet('/content/drive/MyDrive/TFM/favorita_panel_corregido.parquet')
print(esqueleto.shape)
esqueleto.columns.tolist()

Mounted at /content/drive
(4804366, 22)


['store_nbr',
 'item_nbr',
 'city',
 'state',
 'type',
 'cluster',
 'family',
 'class',
 'perishable',
 'date',
 'year',
 'month',
 'day_of_week',
 'is_weekend',
 'es_feriado',
 'dcoilwtico',
 'unit_sales',
 'onpromotion',
 'time_idx',
 'primera_fecha_venta',
 'historia_restante_dias',
 'edad']

# 2. Preparación de variables

In [2]:
if 'type' in esqueleto.columns:
    esqueleto = esqueleto.rename(columns={'type': 'store_type'})

for col in ['store_nbr', 'item_nbr', 'city', 'state', 'store_type', 'cluster', 'class']:
    esqueleto[col] = esqueleto[col].astype(str)

esqueleto.columns.tolist()

['store_nbr',
 'item_nbr',
 'city',
 'state',
 'store_type',
 'cluster',
 'family',
 'class',
 'perishable',
 'date',
 'year',
 'month',
 'day_of_week',
 'is_weekend',
 'es_feriado',
 'dcoilwtico',
 'unit_sales',
 'onpromotion',
 'time_idx',
 'primera_fecha_venta',
 'historia_restante_dias',
 'edad']

# 3. Partición de datos

In [3]:
# División train-test-val
fecha_max = esqueleto['date'].max()
test_inicio = fecha_max - pd.Timedelta(days=60)
val_inicio = test_inicio - pd.Timedelta(days=60)

train = esqueleto[esqueleto['date'] < val_inicio]
val = esqueleto[(esqueleto['date'] >= val_inicio) & (esqueleto['date'] < test_inicio)]

print('Train:', train.shape, '| Val:', val.shape)

Train: (4322665, 22) | Val: (238860, 22)


# 4. Baseline naive estacional

In [4]:
# Se recalcula sobre el panel corregido (28.5% menos filas, 97 series menos)
esqueleto['pred_naive'] = esqueleto.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(7)

test = esqueleto[esqueleto['date'] >= test_inicio].copy()
mae_naive = (test['unit_sales'] - test['pred_naive']).abs().mean()
rmse_naive = ((test['unit_sales'] - test['pred_naive']) ** 2).mean() ** 0.5
wape_naive = (test['unit_sales'] - test['pred_naive']).abs().sum() / test['unit_sales'].sum()

print('Baseline (panel corregido) -> MAE:', mae_naive, '| RMSE:', rmse_naive, '| WAPE:', wape_naive)

Baseline (panel corregido) -> MAE: 3.3928826 | RMSE: 6.218797 | WAPE: 0.55384845


# 5. Instalación e importaciones de PyTorch Forecasting

In [5]:
!pip install pytorch-forecasting pytorch-lightning lightning -q

from pytorch_forecasting import TimeSeriesDataSet, DeepAR
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import NegativeBinomialDistributionLoss
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
import torch

print(torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.4 MB/s eta 0:00:00
True


# 6. Ponderación por volumen

In [6]:
# Se fija un peso inverso al volumen para compensar
# Razón de 85x entre el 10% con más venta y el 10% con menos, según el diagnóstico complementario
# Se usa 1/sqrt en lugar de de 1/volumen puro para suavizar el efecto en series de volumen casi cero.
volumen_por_serie = esqueleto.groupby(['store_nbr', 'item_nbr'])['unit_sales'].transform('mean')
esqueleto['peso_muestra'] = 1 / np.sqrt(volumen_por_serie + 0.1)
esqueleto['peso_muestra'] = esqueleto['peso_muestra'] / esqueleto['peso_muestra'].mean()

esqueleto['peso_muestra'].describe()

,peso_muestra
count,4.804366e+06
mean,1.000000e+00
std,6.774762e-01
min,1.678605e-01
25%,6.279200e-01
50%,8.589107e-01
75%,1.153558e+00
max,6.221284e+00


# 7. Construcción de TimeSeriesDataSet corregido

In [7]:
# Cambios realizados: target_normalizer sin log1p (center=False, para
# NegativeBinomialDistributionLoss), edad añadida a time_varying_known_reals,
# y peso_muestra como columna de ponderación.
max_encoder_length = 90
max_prediction_length = 30
training_cutoff = train['time_idx'].max()

training = TimeSeriesDataSet(
    esqueleto[esqueleto.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="unit_sales",
    group_ids=["store_nbr", "item_nbr"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["city", "state", "store_type", "cluster", "class"],
    time_varying_known_reals=["time_idx", "year", "month", "day_of_week", "is_weekend",
                               "es_feriado", "onpromotion", "edad"],
    time_varying_unknown_reals=["unit_sales"],
    target_normalizer=GroupNormalizer(groups=["store_nbr", "item_nbr"], center=False),
    weight="peso_muestra",
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training, esqueleto, min_prediction_idx=training_cutoff + 1, stop_randomization=True
)

print(training)

/usr/local/lib/python3.13/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 88 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__store_nbr': '1', '__group_id__item_nbr': '1428779'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2002136'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2027777'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2027827'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053590'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053610'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053614'}, {'__group_id__store_nbr': '10', '__group_id__item_nbr': '2033805'}, {'__group_id__store_nbr': '10', '__group_id__item_nbr': '2053590'}, {'__group_id__store_nbr': '1

TimeSeriesDataSet[length=3854726](
	time_idx='time_idx',
	target='unit_sales',
	group_ids=['store_nbr', 'item_nbr'],
	weight='peso_muestra',
	max_encoder_length=90,
	min_encoder_length=90,
	min_prediction_idx=0,
	min_prediction_length=30,
	max_prediction_length=30,
	static_categoricals=['city', 'state', 'store_type', 'cluster', 'class'],
	static_reals=None,
	time_varying_known_categoricals=None,
	time_varying_known_reals=['time_idx', 'year', 'month', 'day_of_week', 'is_weekend', 'es_feriado', 'onpromotion', 'edad'],
	time_varying_unknown_categoricals=None,
	time_varying_unknown_reals=['unit_sales'],
	variable_groups=None,
	constant_fill_strategy=None,
	allow_missing_timesteps=False,
	lags=None,
	add_relative_time_idx=True,
	add_target_scales=True,
	add_encoder_length=True,
	target_normalizer=GroupNormalizer(
	method='standard',
	groups=['store_nbr', 'item_nbr'],
	center=False,
	scale_by_group=False,
	transformation=None,
	method_kwargs={}
),
	categorical_encoders={'__group_id__store_nb

# 8. Dataloaders

In [8]:
batch_size = 128
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

# 9. DeepAR - V7
### 9.1. Configuración inicial (hidden_size=30, batches=200), con binomial negativa

In [9]:
# Se desactivan las logging_metrics por defecto (MAE, RMSE, etc.) por incompatibilidad con weight en esta versión de librería
deepar_v7 = DeepAR.from_dataset(
    training, learning_rate=0.03, hidden_size=30, rnn_layers=2,
    loss=NegativeBinomialDistributionLoss(), logging_metrics=torch.nn.ModuleList([]),
)

trainer_v7 = pl.Trainer(
    max_epochs=30, accelerator="auto", enable_model_summary=True,
    callbacks=[EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=5, mode="min")],
    gradient_clip_val=0.1, limit_train_batches=200, limit_val_batches=50,
)
trainer_v7.fit(deepar_v7, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
trainer_v7.save_checkpoint('/content/drive/MyDrive/TFM/deepar_v7_corregido.ckpt')

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                   ┃ Type                             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                   │ NegativeBinomialDistributionLoss │      0 │ train │     0 │
│ 1 │ logging_metrics        │ ModuleList                       │      0 │ train │     0 │
│ 2 │ embeddings             │ MultiEmbedding                   │    273 │ train │     0 │
│ 3 │ rnn                    │ LSTM                             │ 15.2 K │ train │     0 │
│ 4 │ distribution_projector │ Linear                           │     62 │ train │     0 │
└───┴────────────────────────┴──────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 15.6 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 15.6 K                                                                                               
Total estimated model params size (MB): 0.062                                                                      
Modules in train mode: 11                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `weights_only` was not set, defaulting to `False`.
INFO:lightning.pytorch.trainer.connectors.checkpoint_connector:`weights_only` was not set, defaulting to `False`.
